In [4]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

# tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
# model = AutoModelForCausalLM.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")

# # Initializing a model (with random weights) from the EleutherAI/gpt-neo-1.3B style configuration
# model = GPTNeoForCausalLM(model.config)

# num_params = sum(p.numel() for p in model.parameters())
# print(f"Number of parameters in model: {num_params}")

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")
else:
    print("Warning: HF_TOKEN not found in .env file")

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace


In [8]:
# ============================================
# TEXT GENERATION / INFERENCE
# ============================================

def load_model_for_inference(repo_name=None, checkpoint_step=None, device='cuda'):
    """
    Load a trained model from HuggingFace for text generation
    
    Args:
        repo_name: Full HuggingFace repo name (e.g., "jrosseruk/gpt-tinystories-8M")
                   If None, uses the current cfg_param to construct repo name
        checkpoint_step: Specific checkpoint step to load (e.g., 1000, 2000)
                        If None, loads the latest model from main branch
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The loaded model
        tokenizer: The tokenizer
    """
    if repo_name is None:
        repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    # Determine subfolder if checkpoint_step is specified
    subfolder = None
    if checkpoint_step is not None:
        subfolder = f"checkpoint-{checkpoint_step}"
        print(f"Loading model from {repo_name}/{subfolder}...")
    else:
        print(f"Loading latest model from {repo_name}...")
    
    try:
        from huggingface_hub import repo_exists, list_repo_files
        
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            print(f"Please check the repository name or train a model first")
            return None, None
        
        # If loading a specific checkpoint, verify it exists
        if subfolder is not None:
            try:
                files = list_repo_files(repo_id=repo_name)
                checkpoint_files = [f for f in files if f.startswith(subfolder + '/')]
                
                if not checkpoint_files:
                    print(f"Error: Checkpoint {subfolder} not found in {repo_name}")
                    available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
                    if available_checkpoints:
                        print(f"Available checkpoints: {', '.join(available_checkpoints)}")
                    else:
                        print("No checkpoints found in repository")
                    return None, None
            except Exception as e:
                print(f"Warning: Could not verify checkpoint existence: {e}")
        
        # Load model and tokenizer
        # If subfolder is specified, load from that checkpoint folder
        if subfolder is not None:
            inference_model = GPTNeoForCausalLM.from_pretrained(
                repo_name,
                subfolder=subfolder
            )
        else:
            inference_model = GPTNeoForCausalLM.from_pretrained(repo_name)
        
        inference_tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
        inference_tokenizer.pad_token = inference_tokenizer.eos_token
        
        # Move to device and set to eval mode
        inference_model = inference_model.to(device)
        inference_model.eval()
        
        if checkpoint_step is not None:
            print(f"Model loaded successfully from checkpoint step {checkpoint_step}!")
        else:
            print(f"Model loaded successfully!")
        return inference_model, inference_tokenizer
    
    except FileNotFoundError as e:
        print(f"Error: Could not find required files in {repo_name}")
        print(f"Details: {e}")
        return None, None
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None, None


def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.8, top_k=50, top_p=0.95, num_return_sequences=1, device='cuda'):
    """
    Generate text completion from a prompt
    
    Args:
        model: The trained model
        tokenizer: The tokenizer
        prompt: Text prompt to complete
        max_length: Maximum length of generated text (in tokens)
        temperature: Sampling temperature (higher = more random, lower = more deterministic)
        top_k: Keep only top k tokens with highest probability (0 = disabled)
        top_p: Nucleus sampling - keep top tokens with cumulative probability >= top_p
        num_return_sequences: Number of different completions to generate
        device: Device model is on
    
    Returns:
        List of generated text completions
    """
    model.eval()
    
    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        # Generate
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=max_length + len(input_ids[0]),
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated sequences
    generated_texts = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts


# ============================================
# EXAMPLE USAGE
# ============================================

# Uncomment to load a model and generate text:

# Load from checkpoint 1000
inference_model, inference_tokenizer = load_model_for_inference(checkpoint_step=33000)

# Or load the latest model:
# inference_model, inference_tokenizer = load_model_for_inference()

if inference_model is not None:
    # Define a prompt
    prompt = "Victorious"
    
    print(f"Prompt: {prompt}")
    print("=" * 50)
    
    # Generate completions
    completions = generate_text(
        inference_model, 
        inference_tokenizer, 
        prompt, 
        max_length=150,
        temperature=0.8,
        num_return_sequences=3  # Generate 3 different completions
    )
    
    # Print results
    for i, completion in enumerate(completions, 1):
        print(f"\nCompletion {i}:")
        print(completion)
        print("=" * 50)


print("Text generation functions loaded. Uncomment the example usage block to test!")

Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-33000...
Model loaded successfully from checkpoint step 33000!
Prompt: Victorious

Completion 1:
Victorious was a brave little girl. She wanted to explore the world and ask her mom lots of questions about the world. Every day she would ask her mom what she should be.

One day, she found a big, scary cave. She was scared, but she didn't want to be scared. She wanted to tell her mom, so she put on her coat and went into the cave. She said, "Mom, what is this? I'm scared!"

Just then, her mom walked in and saw what was going on. She said, "Hello, young lady. I'm here to encourage you to explore the cave."

The lady said, "That's very brave of you, young lady. What are you

Completion 2:
Victorious was walking in the park when she saw something black. She ran over to investigate and saw a big dark figure. It was so big, it was scary! 

Suddenly, a voice said, "Stop, stop!" It was a big, black bird.

"What are you doing here?" the b